# User-to-Item Recommendation System Bag-of-Words
In this notebook we created another model that can be used as a lexical baseline which represents each wine using raw token counts.

In [29]:
from pathlib import Path
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
wines_features = pd.read_csv(
    Path("../data/processed/wines_features.csv"),
    usecols=["WineID", "WineName", "Type", "Country", "RegionName", "Body", "Acidity", "content_text"]
)

ratings = pd.read_csv(
    Path("../dataset/XWines_Full_21M_ratings.csv"),
    usecols=["UserID", "WineID", "Rating"],
    dtype={"UserID": "int32", "WineID": "int32", "Rating": "float32"}
)

wines_features = wines_features.drop_duplicates(subset="WineID").copy()
wines_features = wines_features[wines_features["content_text"].notna()].copy()
wines_features = wines_features[wines_features["content_text"].str.strip() != ""].copy()

wines_features["WineID"] = wines_features["WineID"].astype("int32")
ratings["WineID"] = ratings["WineID"].astype("int32")

print("wines_features:", wines_features.shape)
print("ratings:", ratings.shape)

wines_features: (100646, 8)
ratings: (21013536, 3)


In [31]:
from pathlib import Path

RANDOM_STATE = 42
TEST_FRACTION = 0.2

PROJECT_ROOT = Path.cwd().resolve()
for c in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (c / 'data' / 'results' / 'shared_split').exists():
        PROJECT_ROOT = c
        break

SPLIT_DIR = PROJECT_ROOT / 'data' / 'results' / 'shared_split'
ARMS_DIR = PROJECT_ROOT / 'bandits' / 'saved_arms'
ARMS_DIR.mkdir(parents=True, exist_ok=True)

train_pos = pd.read_csv(
    SPLIT_DIR / 'train_pos.csv',
    dtype={'UserID': 'int32', 'WineID': 'int32', 'Rating': 'float32'}
)
test_pos = pd.read_csv(
    SPLIT_DIR / 'test_pos.csv',
    dtype={'UserID': 'int32', 'WineID': 'int32', 'Rating': 'float32'}
)

shared_users = sorted(test_pos['UserID'].unique())

print("Loaded shared split from:", SPLIT_DIR)
print("train_pos:", train_pos.shape)
print("test_pos:", test_pos.shape)
print("shared users:", len(shared_users))

Loaded shared split from: /Users/sofiiaavetisian/Desktop/UNI/recsys/FINAL_WINE/wine-recommendation-system/data/results/shared_split
train_pos: (56673, 3)
test_pos: (16646, 3)
shared users: 5000


In [32]:
bow = CountVectorizer()
bow_matrix = bow.fit_transform(wines_features["content_text"])

print(bow_matrix.shape)

(100646, 93506)


In [33]:
wines_features = wines_features.reset_index(drop=True)

wineid_to_idx = pd.Series(wines_features.index, index=wines_features["WineID"]).drop_duplicates()
idx_to_wineid = pd.Series(wines_features["WineID"].values, index=wines_features.index)

train_pos = train_pos[train_pos["WineID"].isin(wineid_to_idx.index)].copy()
test_pos = test_pos[test_pos["WineID"].isin(wineid_to_idx.index)].copy()

print("train_pos after filtering:", train_pos.shape)
print("test_pos after filtering:", test_pos.shape)

train_pos after filtering: (56673, 3)
test_pos after filtering: (16646, 3)


For each user, positive interactions are divided into training and test sets. The training interactions are used to build the user profile, while the test interactions are used to evaluate whether the model can recover relevant unseen wines.

In [34]:
def build_user_profile_bow(user_history):
    user_history = user_history[user_history["WineID"].isin(wineid_to_idx.index)].copy()

    item_indices = [wineid_to_idx[wine_id] for wine_id in user_history["WineID"]]
    weights = (user_history["Rating"] - 3.0).clip(lower=1.0).to_numpy()

    item_matrix = bow_matrix[item_indices]
    weighted_sum = item_matrix.multiply(weights.reshape(-1, 1)).sum(axis=0)
    profile = weighted_sum / weights.sum()

    return csr_matrix(profile)

In [35]:
user_profiles_bow = {}
user_groups = list(train_pos.groupby("UserID"))

for i, (user_id, user_history) in enumerate(user_groups, start=1):
    user_profiles_bow[user_id] = build_user_profile_bow(user_history)
    if i % 500 == 0:
        print(f"Built {i}/{len(user_groups)} BoW user profiles")

print("built profiles:", len(user_profiles_bow))

Built 500/5000 BoW user profiles
Built 1000/5000 BoW user profiles
Built 1500/5000 BoW user profiles
Built 2000/5000 BoW user profiles
Built 2500/5000 BoW user profiles
Built 3000/5000 BoW user profiles
Built 3500/5000 BoW user profiles
Built 4000/5000 BoW user profiles
Built 4500/5000 BoW user profiles
Built 5000/5000 BoW user profiles
built profiles: 5000


We converted the descriptions into bag of words vectors where each wine is represented by token counts. Unlike the TF-IDF one, here we don't downweight frequent words so common descriptors may have a stronger influence on the similarity.

In [36]:
train_seen = train_pos.groupby("UserID")["WineID"].apply(set).to_dict()
test_relevant = test_pos.groupby("UserID")["WineID"].apply(set).to_dict()

eval_users = sorted(set(user_profiles_bow.keys()) & set(test_relevant.keys()))
print("evaluation users:", len(eval_users))

evaluation users: 5000


In [37]:
def recommend_user_bow_ids(user_id, top_k=10):
    if user_id not in user_profiles_bow:
        return []

    profile = user_profiles_bow[user_id]
    scores = cosine_similarity(profile, bow_matrix).flatten()

    seen_items = train_seen.get(user_id, set())
    ranked_indices = np.argsort(scores)[::-1]

    recommendations = []
    for idx in ranked_indices:
        wine_id = idx_to_wineid[idx]
        if wine_id in seen_items:
            continue
        recommendations.append(wine_id)
        if len(recommendations) == top_k:
            break

    return recommendations

In [38]:
def show_user_recommendations_bow(user_id, top_k=10):
    rec_ids = recommend_user_bow_ids(user_id, top_k=top_k)

    recs = wines_features[wines_features["WineID"].isin(rec_ids)][
        ["WineID", "WineName", "Type", "Country", "RegionName", "Body", "Acidity"]
    ].copy()

    recs["rank"] = recs["WineID"].map({wine_id: i + 1 for i, wine_id in enumerate(rec_ids)})
    recs = recs.sort_values("rank").reset_index(drop=True)

    return recs

In [39]:
train_seen = train_pos.groupby("UserID")["WineID"].apply(set).to_dict()
test_relevant = test_pos.groupby("UserID")["WineID"].apply(set).to_dict()

eval_users = sorted(set(user_profiles_bow.keys()) & set(test_relevant.keys()))
print("evaluation users:", len(eval_users))

evaluation users: 5000


In [40]:
sample_user = eval_users[0]
show_user_recommendations_bow(sample_user, top_k=10)

,WineID,WineName,Type,Country,RegionName,Body,Acidity,rank
0,141223,Nebbiolo d'Alba,Red,Italy,Piemonte,Very full-bodied,High,1
1,138245,Langhe Nebbiolo,Red,Italy,Piemonte,Very full-bodied,High,2
2,143908,Nebbiolo d'Alba,Red,Italy,Barolo,Very full-bodied,High,3
3,146473,Nebbiolo,Red,Italy,Piemonte,Very full-bodied,High,4
4,137533,Michet Nebbiolo d'Alba,Red,Italy,Barolo,Very full-bodied,High,5
5,143878,Roccheri Nebbiolo d'Alba,Red,Italy,Barolo,Very full-bodied,High,6
6,136769,Ornato Barolo,Red,Italy,Piemonte,Very full-bodied,High,7
7,141985,Langhe Nebbiolo,Red,Italy,Piemonte,Very full-bodied,High,8
8,139721,Nebbiolo d'Alba,Red,Italy,Piemonte,Very full-bodied,High,9
9,148977,Nebbiolo d'Alba,Red,Italy,Piemonte,Very full-bodied,High,10


The recsys seems to be capturing a very strong preference but the recommendations might be too narrow for users. It may seem like the same product is being recommended over and over again, since the wineID differs, we treat them as different products. 

In [41]:
def inspect_user_case(user_id, top_k=10):
    print("USER TRAINING WINES")
    display(
        wines_features[wines_features["WineID"].isin(train_seen[user_id])][
            ["WineID", "WineName", "Type", "Country", "RegionName", "Body", "Acidity"]
        ].reset_index(drop=True)
    )

    print("USER HELD-OUT TEST WINES")
    display(
        wines_features[wines_features["WineID"].isin(test_relevant[user_id])][
            ["WineID", "WineName", "Type", "Country", "RegionName", "Body", "Acidity"]
        ].reset_index(drop=True)
    )

    print("RECOMMENDED WINES")
    display(show_user_recommendations_bow(user_id, top_k=top_k))

In [42]:
inspect_user_case(sample_user, top_k=10)

USER TRAINING WINES


,WineID,WineName,Type,Country,RegionName,Body,Acidity
0,111954,Cuvée Prestige Brut Champagne,Sparkling,France,Champagne,Medium-bodied,High
1,117765,Crémant d'Alsace Brut,Sparkling,France,Crémant d'Alsace,Light-bodied,High
2,135957,Ruvei Barbera d'Alba,Red,Italy,Barolo,Medium-bodied,High
3,135993,Trento Brut,Sparkling,Italy,Trento,Medium-bodied,Medium
4,136069,Bricco dell'Uccellone Barbera d'Asti,Red,Italy,Barbera d'Asti,Medium-bodied,High
5,136285,Nebbiolo,Red,Italy,Piemonte,Very full-bodied,High
6,136390,Torre del Falasco Amarone della Valpolicella,Red,Italy,Verona,Very full-bodied,High
7,137093,La Firma Aglianico del Vulture,Red,Italy,Basilicata,Full-bodied,High
8,137339,Aulente Rosso,Red,Italy,Emilia-Romagna,Full-bodied,High
9,137659,Cuvée Imperiale Max Rosé,Sparkling,Italy,Franciacorta,Light-bodied,Medium


USER HELD-OUT TEST WINES


,WineID,WineName,Type,Country,RegionName,Body,Acidity
0,111585,Carte d'Or Brut Champagne,Sparkling,France,Champagne,Medium-bodied,High
1,136020,Nebbiolo Langhe,Red,Italy,Langhe,Very full-bodied,High
2,137683,Vigna di Isalle Cannonau di Sardegna,Red,Italy,Cannonau di Sardegna,Very full-bodied,Low
3,140289,Langhe Nebbiolo,Red,Italy,Langhe,Very full-bodied,High
4,150064,Coste della Sesia Nebbiolo,Red,Italy,Piemonte,Very full-bodied,High


RECOMMENDED WINES


,WineID,WineName,Type,Country,RegionName,Body,Acidity,rank
0,141223,Nebbiolo d'Alba,Red,Italy,Piemonte,Very full-bodied,High,1
1,138245,Langhe Nebbiolo,Red,Italy,Piemonte,Very full-bodied,High,2
2,143908,Nebbiolo d'Alba,Red,Italy,Barolo,Very full-bodied,High,3
3,146473,Nebbiolo,Red,Italy,Piemonte,Very full-bodied,High,4
4,137533,Michet Nebbiolo d'Alba,Red,Italy,Barolo,Very full-bodied,High,5
5,143878,Roccheri Nebbiolo d'Alba,Red,Italy,Barolo,Very full-bodied,High,6
6,136769,Ornato Barolo,Red,Italy,Piemonte,Very full-bodied,High,7
7,141985,Langhe Nebbiolo,Red,Italy,Piemonte,Very full-bodied,High,8
8,139721,Nebbiolo d'Alba,Red,Italy,Piemonte,Very full-bodied,High,9
9,148977,Nebbiolo d'Alba,Red,Italy,Piemonte,Very full-bodied,High,10


In [43]:
user_train_wines = wines_features[wines_features["WineID"].isin(train_seen[sample_user])].copy()

user_train_wines["Country"].value_counts()

Country
Italy     17
France     2
Name: count, dtype: int64

At first it seemed odd that all of the top recommendations were from France, but as we inspected the distribution of the rated wines by country we observed that almost have of the sample user's ratings were for wines from France. 

In [44]:
def precision_at_k(relevant, recommended, k):
    recommended_k = recommended[:k]
    if k == 0:
        return 0.0
    hits = sum(1 for item in recommended_k if item in relevant)
    return hits / k

def recall_at_k(relevant, recommended, k):
    if len(relevant) == 0:
        return 0.0
    recommended_k = recommended[:k]
    hits = sum(1 for item in recommended_k if item in relevant)
    return hits / len(relevant)

def dcg_at_k(relevant, recommended, k):
    score = 0.0
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            score += 1.0 / np.log2(i + 1)
    return score

def ndcg_at_k(relevant, recommended, k):
    ideal_hits = min(len(relevant), k)
    if ideal_hits == 0:
        return 0.0
    ideal_dcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg_at_k(relevant, recommended, k) / ideal_dcg

def average_precision_at_k(relevant, recommended, k):
    recommended_k = recommended[:k]
    hits = 0
    score = 0.0
    for i, item in enumerate(recommended_k, start=1):
        if item in relevant:
            hits += 1
            score += hits / i
    denom = min(len(relevant), k)
    return score / denom if denom > 0 else 0.0

def hit_rate_at_k(relevant, recommended, k):
    recommended_k = recommended[:k]
    return float(any(item in relevant for item in recommended_k))

def reciprocal_rank_at_k(relevant, recommended, k):
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            return 1.0 / i
    return 0.0

In [45]:
def evaluate_recommender(recommend_func, users, test_relevant_dict, ks=(5, 10, 20)):
    rows = []
    max_k = max(ks)

    for i, user_id in enumerate(users, start=1):
        relevant = test_relevant_dict.get(user_id, set())
        if not relevant:
            continue

        recommended = recommend_func(user_id, top_k=max_k)

        row = {"UserID": user_id}
        for k in ks:
            row[f"Precision@{k}"] = precision_at_k(relevant, recommended, k)
            row[f"Recall@{k}"] = recall_at_k(relevant, recommended, k)
            row[f"NDCG@{k}"] = ndcg_at_k(relevant, recommended, k)
            row[f"MAP@{k}"] = average_precision_at_k(relevant, recommended, k)

        rows.append(row)

        if i % 500 == 0:
            print(f"Evaluated {i}/{len(users)} users")

    return pd.DataFrame(rows)

In [46]:
bow_eval = evaluate_recommender(
    recommend_func=recommend_user_bow_ids,
    users=eval_users,
    test_relevant_dict=test_relevant,
    ks=(5, 10, 20)
)

bow_summary = (
    bow_eval
    .drop(columns="UserID")
    .mean()
    .round(4)
    .to_frame(name="BoW")
)

bow_summary

Evaluated 500/5000 users
Evaluated 1000/5000 users
Evaluated 1500/5000 users
Evaluated 2000/5000 users
Evaluated 2500/5000 users
Evaluated 3000/5000 users
Evaluated 3500/5000 users
Evaluated 4000/5000 users
Evaluated 4500/5000 users
Evaluated 5000/5000 users


,BoW
Precision@5,0.0028
Recall@5,0.0060
NDCG@5,0.0047
MAP@5,0.0029
Precision@10,0.0020
Recall@10,0.0076
NDCG@10,0.0054
MAP@10,0.0031
Precision@20,0.0016
Recall@20,0.0113


In [47]:
Path("../data/results").mkdir(parents=True, exist_ok=True)
bow_summary.to_csv("../data/results/bow_summary.csv")

In [48]:
wines_features["eval_signature"] = (
    wines_features["Type"].astype(str).str.lower().str.strip() + "|" +
    wines_features["Country"].astype(str).str.lower().str.strip() + "|" +
    wines_features["RegionName"].astype(str).str.lower().str.strip() + "|" +
    wines_features["Body"].astype(str).str.lower().str.strip() + "|" +
    wines_features["Acidity"].astype(str).str.lower().str.strip()
)

wineid_to_signature = dict(zip(wines_features["WineID"], wines_features["eval_signature"]))

test_relevant_signatures = {
    user_id: {wineid_to_signature[w] for w in wine_ids if w in wineid_to_signature}
    for user_id, wine_ids in test_relevant.items()
}

In [49]:
def recommended_ids_to_signatures(recommended_ids, top_k=None):
    signatures = []
    seen_signatures = set()

    for wine_id in recommended_ids:
        if wine_id not in wineid_to_signature:
            continue

        signature = wineid_to_signature[wine_id]
        if signature in seen_signatures:
            continue

        signatures.append(signature)
        seen_signatures.add(signature)

        if top_k is not None and len(signatures) == top_k:
            break

    return signatures

In [50]:
def evaluate_recommender_soft(recommend_func, users, test_relevant_signatures_dict, ks=(5, 10, 20), oversample_factor=5):
    rows = []
    max_k = max(ks)

    for i, user_id in enumerate(users, start=1):
        relevant = test_relevant_signatures_dict.get(user_id, set())
        if not relevant:
            continue

        recommended_ids = recommend_func(user_id, top_k=max_k * oversample_factor)
        recommended = recommended_ids_to_signatures(recommended_ids, top_k=max_k)

        row = {"UserID": user_id}
        for k in ks:
            row[f"Precision@{k}"] = precision_at_k(relevant, recommended, k)
            row[f"Recall@{k}"] = recall_at_k(relevant, recommended, k)
            row[f"NDCG@{k}"] = ndcg_at_k(relevant, recommended, k)
            row[f"MAP@{k}"] = average_precision_at_k(relevant, recommended, k)
            row[f"HitRate@{k}"] = hit_rate_at_k(relevant, recommended, k)
            row[f"MRR@{k}"] = reciprocal_rank_at_k(relevant, recommended, k)

        rows.append(row)

        if i % 500 == 0:
            print(f"Soft-evaluated {i}/{len(users)} users")

    return pd.DataFrame(rows)

In [51]:
bow_eval_soft = evaluate_recommender_soft(
    recommend_func=recommend_user_bow_ids,
    users=eval_users,
    test_relevant_signatures_dict=test_relevant_signatures,
    ks=(5, 10, 20),
    oversample_factor=5
)

bow_summary_soft = (
    bow_eval_soft
    .drop(columns="UserID")
    .mean()
    .round(4)
    .to_frame(name="BoW Soft")
)

bow_summary_soft

Soft-evaluated 500/5000 users
Soft-evaluated 1000/5000 users
Soft-evaluated 1500/5000 users
Soft-evaluated 2000/5000 users
Soft-evaluated 2500/5000 users
Soft-evaluated 3000/5000 users
Soft-evaluated 3500/5000 users
Soft-evaluated 4000/5000 users
Soft-evaluated 4500/5000 users
Soft-evaluated 5000/5000 users


,BoW Soft
Precision@5,0.0596
Recall@5,0.1096
NDCG@5,0.1166
MAP@5,0.0848
HitRate@5,0.2596
MRR@5,0.1915
Precision@10,0.0343
Recall@10,0.1247
NDCG@10,0.1192
MAP@10,0.0837


In [52]:
Path("../data/results").mkdir(parents=True, exist_ok=True)
bow_eval_soft.to_csv("../data/results/bow_eval_soft.csv", index=False)
bow_summary_soft.to_csv("../data/results/bow_summary_soft.csv")

# Save the model

In [53]:
from pathlib import Path
import pandas as pd

ARMS_DIR = Path("../bandits/saved_arms")
ARMS_DIR.mkdir(parents=True, exist_ok=True)

def export_arm_recs(recommend_fn, users, out_csv, top_k=100):
    rows = []
    for uid in users:
        recs = recommend_fn(int(uid), top_k=top_k)
        for r, wid in enumerate(recs, start=1):
            rows.append({"UserID": int(uid), "rank": int(r), "WineID": int(wid)})
    out_df = pd.DataFrame(rows)
    out_df.to_csv(out_csv, index=False)
    print(f"Saved {len(out_df):,} rows -> {out_csv}")


In [54]:
export_arm_recs(
    recommend_fn=recommend_user_bow_ids,
    users=shared_users,
    out_csv=ARMS_DIR / "content_bow_recs.csv",
    top_k=100
)


Saved 500,000 rows -> ../bandits/saved_arms/content_bow_recs.csv
